# SPECS Analíticas e Dashboards Executivos

Este notebook começa na camada analítica. A `base_consolidada.parquet` é consumida como SOT (Source of Truth); as bases em `dados/bases_analiticas/` são SPECS derivadas para análise e BI.

Fluxo: **SOT → SPECS → indicadores → dashboards executivos**.

In [3]:
from pathlib import Path
import re
import pandas as pd

BASE_PROJETO = Path.cwd()
ORIGEM_SOT = BASE_PROJETO / 'dados' / 'base_consolidada.parquet'
SAIDA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'
if not ORIGEM_SOT.exists():
    BASE_PROJETO = Path('d:/d/pos/Fase 3')
    ORIGEM_SOT = BASE_PROJETO / 'dados' / 'base_consolidada.parquet'
    SAIDA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'
assert ORIGEM_SOT.exists(), f'SOT não encontrada: {ORIGEM_SOT}'
SAIDA_SPECS.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ORIGEM_SOT)
df['id'] = df['id'].astype('string')
df['respondente_key'] = df['ano_pesquisa'].astype('string') + '_' + df['id'].astype('string')

def ativo(valor):
    return not pd.isna(valor) and str(valor).strip().lower() in {'1', '1.0', 'true', 'sim', 'yes', 'x'}

def itens_multivalorados(valor):
    if pd.isna(valor):
        return []
    invalidos = {'', 'nan', 'não utilizo nenhuma das linguagens listadas', 'nenhuma', 'não se aplica'}
    itens = [re.sub(r'\s+', ' ', item).strip() for item in str(valor).split(',')]
    return list(dict.fromkeys(item for item in itens if item.lower() not in invalidos))

def salvar(nome, base):
    caminho = SAIDA_SPECS / f'{nome}.parquet'
    base.to_parquet(caminho, index=False)
    return caminho

dimensoes = [
    'respondente_key', 'id', 'ano_pesquisa', 'genero', 'cor_raca_etnia',
    'estado_onde_mora', 'uf_onde_mora', 'regiao_onde_mora', 'nivel_de_ensino',
    'area_de_formacao', 'setor', 'cargo_atual', 'nivel', 'faixa_salarial',
    'tempo_experiencia_dados', 'oportunidade_buscada', 'modelo_trabalho_atual',
    'modelo_trabalho_ideal'
]
dimensoes = [coluna for coluna in dimensoes if coluna in df.columns]
segmento_cols = [coluna for coluna in ['ano_pesquisa', 'regiao_onde_mora', 'nivel', 'tempo_experiencia_dados', 'modelo_trabalho_atual'] if coluna in df.columns]
spec_respondentes = df[dimensoes].drop_duplicates('respondente_key').copy()

familias_tecnologia = {
    'linguagem': ['sql', 'r', 'python', 'c_cpp_csharp', 'dotnet', 'java', 'julia', 'sas_stata', 'visual_basic_vba', 'scala', 'matlab', 'rust', 'php', 'javascript'],
    'banco_ou_plataforma': ['mysql', 'oracle', 'sql_server', 'amazon_aurora_rds', 'dynamodb', 'cassandra', 'mongodb', 'mariadb', 'postgresql', 's3', 'google_bigquery', 'amazon_redshift', 'amazon_athena', 'snowflake', 'databricks', 'hive', 'firebird'],
    'cloud': ['aws_cloud', 'gcp_cloud', 'azure_cloud', 'oracle_cloud', 'ibm', 'cloud_propria'],
    'etl': ['scripts_python', 'sql_stored_procedures', 'apache_airflow', 'apache_nifi', 'luigi', 'aws_glue', 'talend', 'pentaho', 'alteryx', 'stitch', 'fivetran', 'google_dataflow', 'databricks_etl'],
}

linhas_tecnologia = []
for familia, indicadores in familias_tecnologia.items():
    for coluna in indicadores:
        if coluna in df.columns:
            base = df.loc[df[coluna].map(ativo), ['respondente_key', 'id', 'ano_pesquisa', 'regiao_onde_mora', 'nivel', 'tempo_experiencia_dados', 'modelo_trabalho_atual']].copy()
            base['familia'] = familia
            base['tecnologia'] = coluna.replace('_cloud', '').replace('_etl', '').replace('_', ' ').strip().title()
            base['tecnologia_principal'] = False
            linhas_tecnologia.append(base)

colunas_linguagem = ['respondente_key', 'id', 'ano_pesquisa', 'regiao_onde_mora', 'nivel', 'tempo_experiencia_dados', 'modelo_trabalho_atual', 'linguagens_utilizadas', 'linguagem_principal']
for _, registro in df[colunas_linguagem].iterrows():
    for tecnologia in itens_multivalorados(registro['linguagens_utilizadas']):
        tecnologia = tecnologia.casefold().strip().title()
        linhas_tecnologia.append(pd.DataFrame([{
            'respondente_key': registro['respondente_key'], 'id': registro['id'], 'ano_pesquisa': registro['ano_pesquisa'],
            'regiao_onde_mora': registro['regiao_onde_mora'], 'nivel': registro['nivel'],
            'tempo_experiencia_dados': registro['tempo_experiencia_dados'], 'modelo_trabalho_atual': registro['modelo_trabalho_atual'],
            'familia': 'linguagem', 'tecnologia': tecnologia,
            'tecnologia_principal': tecnologia == str(registro['linguagem_principal']).strip().casefold().title(),
        }]))
spec_adocao_tecnologias = pd.concat(linhas_tecnologia, ignore_index=True).drop_duplicates(['respondente_key', 'familia', 'tecnologia'])

indicadores_ia = [
    'ia_independente_descentralizada', 'ia_direcionamento_centralizado', 'desenvolvedores_copilot',
    'ia_produtos_externos', 'ia_produtos_internos', 'ia_principal_frente_negocio', 'utiliza_chatgpt_llms',
    'ia_solucoes_gratuitas', 'ia_paga_usuario', 'ia_paga_empresa', 'usa_copilot',
    'falta_compreensao_casos_uso', 'falta_confiabilidade_saidas', 'incerteza_regulamentacao',
    'preocupacoes_seguranca_privacidade', 'roi_nao_comprovado_ia', 'dados_empresa_nao_prontos_ia',
    'falta_expertise_recursos', 'alta_direcao_nao_ve_valor', 'preocupacoes_propriedade_intelectual',
]
barreiras_ia = {'falta_compreensao_casos_uso', 'falta_confiabilidade_saidas', 'incerteza_regulamentacao', 'preocupacoes_seguranca_privacidade', 'roi_nao_comprovado_ia', 'dados_empresa_nao_prontos_ia', 'falta_expertise_recursos', 'alta_direcao_nao_ve_valor', 'preocupacoes_propriedade_intelectual'}
linhas_ia = []
for coluna in indicadores_ia:
    if coluna in df.columns:
        base = df.loc[df[coluna].map(ativo), ['respondente_key', 'id', 'ano_pesquisa', 'regiao_onde_mora', 'nivel', 'tempo_experiencia_dados', 'modelo_trabalho_atual']].copy()
        base['indicador_ia'] = coluna
        base['valor'] = 1
        base['tipo_barreira'] = coluna if coluna in barreiras_ia else pd.NA
        linhas_ia.append(base)
spec_adocao_ia = pd.concat(linhas_ia, ignore_index=True).drop_duplicates(['respondente_key', 'indicador_ia'])

spec_diversidade_carreira = spec_respondentes.copy()
spec_adocao_ia_uso = spec_adocao_ia[spec_adocao_ia['indicador_ia'].isin([item for item in indicadores_ia if item not in barreiras_ia])].copy()
ia_por_respondente = spec_adocao_ia_uso.groupby('respondente_key').agg(ia_indicadores=('indicador_ia', 'nunique')).reset_index()
spec_segmentacao_negocio = spec_respondentes.merge(ia_por_respondente, on='respondente_key', how='left')
spec_segmentacao_negocio['ia_indicadores'] = spec_segmentacao_negocio['ia_indicadores'].fillna(0)
spec_segmentacao_negocio = spec_segmentacao_negocio.groupby(segmento_cols, dropna=False).agg(
    respondentes=('respondente_key', 'nunique'),
    media_indicadores_ia=('ia_indicadores', 'mean'),
    salario_moda=('faixa_salarial', lambda serie: serie.mode().iat[0] if not serie.mode().empty else pd.NA),
).reset_index()

for nome, base in {'spec_respondentes': spec_respondentes, 'spec_adocao_tecnologias': spec_adocao_tecnologias, 'spec_adocao_ia': spec_adocao_ia, 'spec_diversidade_carreira': spec_diversidade_carreira, 'spec_segmentacao_negocio': spec_segmentacao_negocio}.items():
    salvar(nome, base)

print(f'SOT consumida: {ORIGEM_SOT}')
print(f'SPECS geradas em: {SAIDA_SPECS}')
print(f'Respondentes: {spec_respondentes.respondente_key.nunique():,} | Tecnologias: {len(spec_adocao_tecnologias):,} | IA: {len(spec_adocao_ia):,}')

SOT consumida: d:\d\pos\Fase 3\projeto\fase_3_data_analytics\dados\base_consolidada.parquet
SPECS geradas em: d:\d\pos\Fase 3\projeto\fase_3_data_analytics\dados\bases_analiticas
Respondentes: 14,005 | Tecnologias: 62,446 | IA: 29,019


## Dashboards Executivos

Os indicadores abaixo foram calculados a partir das SPECS. Adoção de IA considera somente uso; barreiras são mantidas para análise de risco e prontidão.

In [4]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import sqlglot

con = duckdb.connect()
for nome, base in {
    "spec_respondentes": spec_respondentes,
    "spec_adocao_tecnologias": spec_adocao_tecnologias,
    "spec_adocao_ia": spec_adocao_ia,
    "spec_diversidade_carreira": spec_diversidade_carreira,
    "spec_segmentacao_negocio": spec_segmentacao_negocio,
}.items():
    con.register(nome, base)
con.register("spec_adocao_ia_uso", spec_adocao_ia_uso)

def executar_sql(consulta):
    consulta_validada = sqlglot.parse_one(consulta, read="duckdb")
    return con.sql(consulta_validada.sql(dialect="duckdb")).df()

total = int(executar_sql("SELECT COUNT(DISTINCT respondente_key) AS total FROM spec_respondentes").iloc[0, 0])
genero = executar_sql("""SELECT COALESCE(genero, 'Nao informado') AS genero, COUNT(DISTINCT respondente_key) AS respondentes, ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes), 1) AS percentual FROM spec_diversidade_carreira GROUP BY 1 ORDER BY respondentes DESC""")
top_tecnologias = executar_sql("""SELECT tecnologia, COUNT(DISTINCT respondente_key) AS respondentes, ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes), 1) AS adocao_pct FROM spec_adocao_tecnologias GROUP BY tecnologia ORDER BY respondentes DESC LIMIT 12""")
ia_resumo = executar_sql("""SELECT indicador_ia, COUNT(DISTINCT respondente_key) AS respondentes, ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes), 1) AS adocao_pct FROM spec_adocao_ia_uso GROUP BY indicador_ia ORDER BY respondentes DESC""")
barreiras = executar_sql("""SELECT tipo_barreira, COUNT(DISTINCT respondente_key) AS ocorrencias FROM spec_adocao_ia WHERE tipo_barreira IS NOT NULL GROUP BY tipo_barreira ORDER BY ocorrencias DESC LIMIT 8""")

def adocao_por_dimensao(coluna):
    base = spec_respondentes.groupby(coluna, dropna=False)["respondente_key"].nunique().rename("respondentes")
    adotantes = spec_respondentes[spec_respondentes["respondente_key"].isin(set(spec_adocao_ia_uso["respondente_key"]))]
    uso = adotantes.groupby(coluna, dropna=False)["respondente_key"].nunique().rename("adotantes")
    resultado = pd.concat([base, uso], axis=1).fillna(0).reset_index()
    resultado["adocao_pct"] = (100 * resultado["adotantes"] / resultado["respondentes"]).round(1)
    resultado["segmento"] = resultado[coluna].fillna("Nao informado")
    return resultado

segmentos_corrigidos = {nome: adocao_por_dimensao(coluna) for nome, coluna in {
    "Região": "regiao_onde_mora", "Senioridade": "nivel", "Modelo de trabalho": "modelo_trabalho_atual",
}.items()}
rotulos = {
    "Modelo 100% presencial": "Presencial",
    "Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)": "Híbrido flexível",
    "Modelo híbrido com dias fixos de trabalho presencial": "Híbrido fixo",
    "Modelo 100% remoto": "Remoto",
    "Especialista/Staff+": "Especialista / Staff+",
    "Centro-oeste": "Centro-Oeste",
}

ia_adotantes = spec_adocao_ia_uso["respondente_key"].nunique()
intensidade_media = spec_adocao_ia_uso.groupby("respondente_key")["indicador_ia"].nunique().mean()
print(f"Respondentes: {total:,} | Adoção de IA: {100 * ia_adotantes / total:.1f}% | Intensidade média: {intensidade_media:.2f} indicadores")

sns.set_theme(style="whitegrid", font_scale=0.9)
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
for eixo, (nome, resultado) in zip(axes, segmentos_corrigidos.items()):
    dados = resultado[resultado["segmento"] != "Nao informado"].sort_values("adocao_pct").copy()
    dados["label"] = dados["segmento"].map(rotulos).fillna(dados["segmento"])
    sns.barplot(data=dados, x="adocao_pct", y="label", ax=eixo, color="#0b7285")
    eixo.set_title(f"IA por {nome.lower()}")
    eixo.set_xlabel("% com uso de IA")
    eixo.set_ylabel("")
    eixo.set_xlim(0, 100)
    for barra, amostra in zip(eixo.patches, dados["respondentes"]):
        eixo.text(barra.get_width() + 1, barra.get_y() + barra.get_height() / 2, f"n={int(amostra):,}", va="center", fontsize=8)
plt.suptitle("Comparação de adoção de IA por grupo", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

print("Nota executiva: os nomes completos dos grupos permanecem disponíveis na tabela de dados; o eixo usa labels curtos para leitura executiva.")

ModuleNotFoundError: No module named 'duckdb'

## Evolução Anual

Volume da amostra, adoção de IA, participação feminina e tecnologias líderes por ano. Os anos são tratados como categorias para evitar rótulos intermediários.

In [ ]:
anos = sorted(spec_respondentes['ano_pesquisa'].dropna().unique())
base_anual = spec_respondentes.groupby('ano_pesquisa')['respondente_key'].nunique().rename('respondentes')
adotantes_anuais = spec_adocao_ia_uso.groupby('ano_pesquisa')['respondente_key'].nunique().rename('adotantes_ia')
mulheres_anuais = spec_respondentes[spec_respondentes['genero'] == 'Feminino'].groupby('ano_pesquisa')['respondente_key'].nunique().rename('mulheres')
evolucao_anual = pd.concat([base_anual, adotantes_anuais, mulheres_anuais], axis=1).fillna(0).reset_index()
evolucao_anual['adocao_ia_pct'] = (100 * evolucao_anual['adotantes_ia'] / evolucao_anual['respondentes']).round(1)
evolucao_anual['participacao_feminina_pct'] = (100 * evolucao_anual['mulheres'] / evolucao_anual['respondentes']).round(1)
evolucao_anual['ano'] = evolucao_anual['ano_pesquisa'].astype(str)
print(evolucao_anual[['ano', 'respondentes', 'adocao_ia_pct', 'participacao_feminina_pct']].to_string(index=False))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Tamanho da amostra', 'Indicadores percentuais'))
fig.add_trace(go.Scatter(x=evolucao_anual['ano'], y=evolucao_anual['respondentes'], mode='lines+markers+text', text=evolucao_anual['respondentes'].map(lambda valor: f'{int(valor):,}'), textposition='top center', name='Respondentes', line={'color': '#0b7285', 'width': 3}), row=1, col=1)
fig.add_trace(go.Scatter(x=evolucao_anual['ano'], y=evolucao_anual['adocao_ia_pct'], mode='lines+markers+text', text=evolucao_anual['adocao_ia_pct'].map(lambda valor: f'{valor:.1f}%'), textposition='top center', name='Adoção de IA', line={'color': '#f08c46', 'width': 3}), row=1, col=2)
fig.add_trace(go.Scatter(x=evolucao_anual['ano'], y=evolucao_anual['participacao_feminina_pct'], mode='lines+markers+text', text=evolucao_anual['participacao_feminina_pct'].map(lambda valor: f'{valor:.1f}%'), textposition='bottom center', name='Participação feminina', line={'color': '#d9485f', 'width': 3}), row=1, col=2)
fig.update_xaxes(type='category', categoryorder='array', categoryarray=[str(ano) for ano in anos])
fig.update_yaxes(range=[0, 100], ticksuffix='%', row=1, col=2)
fig.update_layout(title='Dashboard Executivo | Evolução anual', template='plotly_white', height=480, width=1100, hovermode='x unified', margin={'l': 60, 'r': 35, 't': 90, 'b': 60})
fig.show()

top5 = top_tecnologias.head(5)['tecnologia'].tolist()
tecnologia_anual = spec_adocao_tecnologias[spec_adocao_tecnologias['tecnologia'].isin(top5)].groupby(['ano_pesquisa', 'tecnologia'])['respondente_key'].nunique().reset_index(name='adotantes').merge(base_anual.reset_index().rename(columns={'respondentes': 'total'}), on='ano_pesquisa')
tecnologia_anual['ano'] = tecnologia_anual['ano_pesquisa'].astype(str)
tecnologia_anual['adocao_pct'] = 100 * tecnologia_anual['adotantes'] / tecnologia_anual['total']
fig_tecnologia = px.line(tecnologia_anual, x='ano', y='adocao_pct', color='tecnologia', markers=True, title='Evolução anual das tecnologias líderes', labels={'ano': 'Ano', 'adocao_pct': '% dos respondentes', 'tecnologia': 'Tecnologia'}, height=480, width=1000)
fig_tecnologia.update_layout(template='plotly_white', hovermode='x unified')
fig_tecnologia.update_xaxes(type='category', categoryorder='array', categoryarray=[str(ano) for ano in anos], tickmode='array', tickvals=[str(ano) for ano in anos], ticktext=[str(ano) for ano in anos])
fig_tecnologia.show()

## Resumo para Diretoria

Use o quadro abaixo como roteiro de apresentação: fato observado, implicação para o negócio e decisão recomendada.

In [ ]:
pct_feminino = float(genero.loc[genero["genero"] == "Feminino", "percentual"].iloc[0])
tecnologia_lider = top_tecnologias.iloc[0]["tecnologia"]
tecnologia_lider_pct = float(top_tecnologias.iloc[0]["adocao_pct"])
adocao_2024 = float(evolucao_anual.loc[evolucao_anual["ano_pesquisa"] == 2024, "adocao_ia_pct"].iloc[0])
adocao_2025 = float(evolucao_anual.loc[evolucao_anual["ano_pesquisa"] == 2025, "adocao_ia_pct"].iloc[0])
amostra_2024 = int(evolucao_anual.loc[evolucao_anual["ano_pesquisa"] == 2024, "respondentes"].iloc[0])
amostra_2025 = int(evolucao_anual.loc[evolucao_anual["ano_pesquisa"] == 2025, "respondentes"].iloc[0])
barreira_principal = barreiras.iloc[0]["tipo_barreira"].replace("_", " ").title()

resumo_diretoria = pd.DataFrame([
    {
        "Indicador": "Adoção de IA",
        "Fato observado": f"{100 * ia_adotantes / total:.1f}% no período; {adocao_2024:.1f}% em 2024 e {adocao_2025:.1f}% em 2025.",
        "Implicação": "A IA já está em uso, mas a evolução anual não é contínua.",
        "Decisão recomendada": "Priorizar casos de uso com valor mensurável.",
    },
    {
        "Indicador": "Tecnologia base",
        "Fato observado": f"{tecnologia_lider} lidera com {tecnologia_lider_pct:.1f}%.",
        "Implicação": "Há uma base técnica comum para capacitação e produtividade.",
        "Decisão recomendada": "Consolidar trilhas de SQL, Python e cloud.",
    },
    {
        "Indicador": "Diversidade",
        "Fato observado": f"Mulheres representam {pct_feminino:.1f}% da amostra.",
        "Implicação": "O pipeline de talentos permanece concentrado.",
        "Decisão recomendada": "Acompanhar contratação, retenção e promoção por gênero.",
    },
    {
        "Indicador": "Prontidão para escala",
        "Fato observado": f"Principal barreira: {barreira_principal}.",
        "Implicação": "O gargalo está em capacidade, dados e execução.",
        "Decisão recomendada": "Investir em governança, dados preparados e capacitação.",
    },
    {
        "Indicador": "Comparação anual",
        "Fato observado": f"A amostra de 2025 foi {100 * (1 - amostra_2025 / amostra_2024):.1f}% menor que a de 2024.",
        "Implicação": "A queda de adoção deve ser interpretada com cautela.",
        "Decisão recomendada": "Confirmar a tendência em uma nova medição.",
    },
])

print("MENSAGEM PRINCIPAL")
print("Transformar experimentação de IA em resultado de negócio exige dados confiáveis, governança, capacitação e diversidade.")
display(resumo_diretoria)
print("Escopo: respostas declaradas dos surveys 2023-2025; não medem causalidade, produtividade ou ROI financeiro.")

In [ ]:
# ---------------------------------------------------------
# CÉLULA DE CÓDIGO: DIVERSIDADE E TECNOLOGIAS BASE
# ---------------------------------------------------------
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Criando subplots interativos: 1 linha, 2 colunas
fig_panorama = make_subplots(
    rows=1, cols=2, 
    specs=[[{"type": "domain"}, {"type": "xy"}]],
    subplot_titles=('Cenário de Diversidade de Gênero', 'Top 10 Tecnologias Mais Adotadas')
)

# 1. Gráfico de Rosca: Diversidade de Gênero
fig_panorama.add_trace(
    go.Pie(
        labels=genero['genero'], 
        values=genero['respondentes'], 
        hole=0.45, 
        marker_colors=['#2a3f5f', '#d9485f', '#f08c46'],
        textinfo='label+percent',
        hoverinfo='label+value+percent'
    ),
    row=1, col=1
)

# 2. Gráfico de Barras Horizontais: Tecnologias Base
top10_tech = top_tecnologias.head(10).sort_values('adocao_pct', ascending=True)
fig_panorama.add_trace(
    go.Bar(
        x=top10_tech['adocao_pct'], 
        y=top10_tech['tecnologia'], 
        orientation='h', 
        marker_color='#0b7285',
        text=top10_tech['adocao_pct'].apply(lambda x: f'{x}%'),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

# Ajuste de layout para padrão executivo
fig_panorama.update_layout(
    title={'text': 'Dashboard Executivo | Pessoas e Ferramentas', 'font': {'size': 18}},
    template='plotly_white',
    height=500,
    margin=dict(t=90, b=40, l=40, r=40)
)
fig_panorama.update_xaxes(range=[0, 100], ticksuffix='%', row=1, col=2)

fig_panorama.show()

In [ ]:
print("=========================================================================================================")
print("RESUMO EXECUTIVO E DIRECIONAMENTO ESTRATÉGICO")
print("=========================================================================================================")
print("DESAFIOS:")
print("- A barreira #1 ('Falta de Expertise e Recursos') trava a evolução da IA de um estágio experimental para impacto real.")
print("- Concentração profunda do pipeline de talentos, evidenciada pela baixa diversidade de gênero (23,5%).")
print("\\nOPORTUNIDADES DE INVESTIMENTO:")
print("- O alto índice de uso de IA (64,3%) valida o apetite dos times. É hora de investir em governança e dados limpos.")
print("- Consolidar treinamentos oficiais em SQL e Python garantirá ganhos de produtividade e escalabilidade imediatos.")
print("=========================================================================================================\\n")

display(resumo_diretoria)
print("\\n*Escopo: base auditável do State of Data Brasil 2023-2025. Métricas de IA refletem uso tático, não o ROI financeiro final.*")

In [ ]:
print("=========================================================================================================")
print("RESUMO EXECUTIVO E DIRECIONAMENTO ESTRATÉGICO")
print("=========================================================================================================")
print("DESAFIOS:")
print("- A barreira #1 ('Falta de Expertise e Recursos') trava a evolução da IA de um estágio experimental para impacto real.")
print("- Concentração profunda do pipeline de talentos, evidenciada pela baixa diversidade de gênero (23,5%).")
print("\\nOPORTUNIDADES DE INVESTIMENTO:")
print("- O alto índice de uso de IA (64,3%) valida o apetite dos times. É hora de investir em governança e dados limpos.")
print("- Consolidar treinamentos oficiais em SQL e Python garantirá ganhos de produtividade e escalabilidade imediatos.")
print("=========================================================================================================\\n")

display(resumo_diretoria)
print("\\n*Escopo: base auditável do State of Data Brasil 2023-2025. Métricas de IA refletem uso tático, não o ROI financeiro final.*")

In [ ]:
import duckdb
import pandas as pd
import re
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import sqlglot

# 1. Carregamento da Fonte da Verdade (SOT)
BASE_PROJETO = Path.cwd()
ORIGEM_SOT = BASE_PROJETO / 'dados' / 'base_consolidada.parquet'
SAIDA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'

if not ORIGEM_SOT.exists():
    BASE_PROJETO = Path('d:/d/pos/Fase 3')
    ORIGEM_SOT = BASE_PROJETO / 'dados' / 'base_consolidada.parquet'
    SAIDA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'

assert ORIGEM_SOT.exists(), f'SOT não encontrada: {ORIGEM_SOT}'
SAIDA_SPECS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ORIGEM_SOT)
df['id'] = df['id'].astype('string')
df['respondente_key'] = df['ano_pesquisa'].astype('string') + '_' + df['id'].astype('string')

# 2. Funções auxiliares
def ativo(valor):
    return not pd.isna(valor) and str(valor).strip().lower() in {'1', '1.0', 'true', 'sim', 'yes', 'x'}

def itens_multivalorados(valor):
    if pd.isna(valor):
        return []
    invalidos = {'', 'nan', 'não utilizo nenhuma das linguagens listadas', 'nenhuma', 'não se aplica'}
    itens = [re.sub(r'\s+', ' ', item).strip() for item in str(valor).split(',')]
    return list(dict.fromkeys(item for item in itens if item.lower() not in invalidos))

# 3. Construção das SPECS
dimensoes = [
    'respondente_key', 'ano_pesquisa', 'genero', 'regiao_onde_mora', 
    'nivel', 'faixa_salarial', 'modelo_trabalho_atual'
]
dimensoes = [c for c in dimensoes if c in df.columns]
spec_respondentes = df[dimensoes].drop_duplicates('respondente_key').copy()

familias_tecnologia = {
    'linguagem': ['sql', 'python', 'r', 'java', 'scala', 'javascript'],
    'banco_ou_plataforma': ['postgresql', 'mysql', 'sql_server', 'oracle', 'google_bigquery', 'snowflake', 'databricks'],
    'cloud': ['aws_cloud', 'gcp_cloud', 'azure_cloud'],
}

linhas_tecnologia = []
for familia, indicadores in familias_tecnologia.items():
    for coluna in indicadores:
        if coluna in df.columns:
            base = df.loc[df[coluna].map(ativo), ['respondente_key', 'ano_pesquisa']].copy()
            base['familia'] = familia
            base['tecnologia'] = coluna.replace('_cloud', '').replace('_', ' ').strip().title()
            linhas_tecnologia.append(base)

spec_adocao_tecnologias = pd.concat(linhas_tecnologia, ignore_index=True).drop_duplicates()

indicadores_ia = [
    'ia_independente_descentralizada', 'ia_direcionamento_centralizado', 
    'utiliza_chatgpt_llms', 'usa_copilot', 'falta_expertise_recursos', 
    'dados_empresa_nao_prontos_ia', 'roi_nao_comprovado_ia'
]
barreiras_ia = {'falta_expertise_recursos', 'dados_empresa_nao_prontos_ia', 'roi_nao_comprovado_ia'}

linhas_ia = []
for coluna in indicadores_ia:
    if coluna in df.columns:
        base = df.loc[df[coluna].map(ativo), ['respondente_key', 'ano_pesquisa']].copy()
        base['indicador_ia'] = coluna
        base['tipo_barreira'] = coluna if coluna in barreiras_ia else pd.NA
        linhas_ia.append(base)

spec_adocao_ia = pd.concat(linhas_ia, ignore_index=True)
spec_adocao_ia_uso = spec_adocao_ia[spec_adocao_ia['tipo_barreira'].isna()].copy()

# 4. DuckDB e Agregações base
con = duckdb.connect()
con.register("spec_respondentes", spec_respondentes)
con.register("spec_adocao_tecnologias", spec_adocao_tecnologias)
con.register("spec_adocao_ia", spec_adocao_ia)
con.register("spec_adocao_ia_uso", spec_adocao_ia_uso)

def executar_sql(consulta):
    return con.sql(sqlglot.parse_one(consulta, read="duckdb").sql(dialect="duckdb")).df()

total_resp = int(executar_sql("SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes").iloc[0, 0])
print(f"✅ Processamento concluído! Respondentes únicos: {total_resp:,}")

In [ ]:
# Dados
genero = executar_sql("""
    SELECT COALESCE(genero, 'Não informado') AS genero, COUNT(DISTINCT respondente_key) AS qtd
    FROM spec_respondentes GROUP BY 1 ORDER BY qtd DESC
""")
top_tech = executar_sql("""
    SELECT tecnologia, COUNT(DISTINCT respondente_key) AS qtd, 
    ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes), 1) AS pct
    FROM spec_adocao_tecnologias GROUP BY tecnologia ORDER BY qtd DESC LIMIT 8
""")

# Visualização
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]],
                    subplot_titles=('Distribuição de Gênero', 'Top Tecnologias Mais Adotadas (%)'))

fig.add_trace(go.Pie(labels=genero['genero'], values=genero['qtd'], hole=0.5, 
                     marker_colors=['#2a3f5f', '#d9485f', '#C8D4E3']), row=1, col=1)

fig.add_trace(go.Bar(x=top_tech['pct'], y=top_tech['tecnologia'], orientation='h', 
                     marker_color='#0b7285', text=top_tech['pct'].apply(lambda x: f'{x}%'),
                     textposition='outside'), row=1, col=2)

fig.update_layout(template='plotly_white', height=450, showlegend=False, 
                  title={'text': '<b>Panorama Inicial: Quem são e o que usam?</b>', 'font': {'size': 18}})
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_xaxes(range=[0, 70], row=1, col=2)
fig.show()

In [ ]:
ia_adotantes = spec_adocao_ia_uso["respondente_key"].nunique()
intensidade_media = spec_adocao_ia_uso.groupby("respondente_key")["indicador_ia"].nunique().mean()
pct_ia = (ia_adotantes / total_resp) * 100

# Evolução IA
evolucao = executar_sql("""
    SELECT ano_pesquisa::VARCHAR as ano,
    ROUND(100.0 * COUNT(DISTINCT a.respondente_key) / COUNT(DISTINCT r.respondente_key), 1) as adocao_pct
    FROM spec_respondentes r
    LEFT JOIN spec_adocao_ia_uso a ON r.respondente_key = a.respondente_key
    GROUP BY ano_pesquisa ORDER BY ano_pesquisa
""")

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "indicator"}, {"type": "scatter"}]])

# KPI
fig.add_trace(go.Indicator(
    mode="number+gauge", value=pct_ia, number={'suffix': "%", 'font': {'size': 50, 'color': '#f08c46'}},
    title={'text': f"Profissionais utilizando IA<br><span style='font-size:14px;color:gray'>Média de {intensidade_media:.2f} ferramentas/usuário</span>"},
    gauge={'axis': {'range': [0, 100]}, 'bar': {'color': "#f08c46"}, 'bgcolor': "lightgray"}
), row=1, col=1)

# Linha de Evolução
fig.add_trace(go.Scatter(x=evolucao['ano'], y=evolucao['adocao_pct'], mode='lines+markers+text',
                         text=evolucao['adocao_pct'].apply(lambda x: f'{x}%'), textposition='top center',
                         line=dict(color='#0b7285', width=4), marker=dict(size=10)), row=1, col=2)

fig.update_layout(template='plotly_white', height=400, title={'text': '<b>Tração de IA no Fluxo de Trabalho</b>', 'font': {'size': 18}})
fig.update_yaxes(range=[0, 100], title="% Adoção", row=1, col=2)
fig.show()

In [ ]:
def query_segmento(coluna):
    return executar_sql(f"""
        SELECT COALESCE({coluna}, 'N/I') as segmento, 
               COUNT(DISTINCT r.respondente_key) as total,
               COUNT(DISTINCT a.respondente_key) as adotantes,
               ROUND(100.0 * COUNT(DISTINCT a.respondente_key) / COUNT(DISTINCT r.respondente_key), 1) as pct
        FROM spec_respondentes r
        LEFT JOIN spec_adocao_ia_uso a ON r.respondente_key = a.respondente_key
        GROUP BY 1 HAVING COUNT(DISTINCT r.respondente_key) > 50 ORDER BY pct ASC
    """)

df_reg = query_segmento('regiao_onde_mora')
df_niv = query_segmento('nivel')
df_mod = query_segmento('modelo_trabalho_atual')

# Padronizando nomes longos
df_mod['segmento'] = df_mod['segmento'].replace({
    'Modelo 100% presencial': 'Presencial',
    'Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)': 'Híbrido Flexível',
    'Modelo híbrido com dias fixos de trabalho presencial': 'Híbrido Fixo',
    'Modelo 100% remoto': 'Remoto'
})

fig = make_subplots(rows=1, cols=3, subplot_titles=('Por Região', 'Por Senioridade', 'Por Modelo de Trabalho'),
                    horizontal_spacing=0.1)

def add_bar(df, row, col):
    fig.add_trace(go.Bar(
        x=df['pct'], y=df['segmento'], orientation='h', marker_color='#2a3f5f',
        text=df.apply(lambda row: f"{row['pct']}% (n={row['total']})", axis=1), textposition='inside', insidetextanchor='end'
    ), row=row, col=col)

add_bar(df_reg, 1, 1)
add_bar(df_niv, 1, 2)
add_bar(df_mod, 1, 3)

fig.update_layout(template='plotly_white', height=450, showlegend=False, title={'text': '<b>Adoção de IA por Demografia e Rotina</b>', 'font': {'size': 18}})
fig.update_xaxes(range=[0, 100])
fig.show()

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "intro",
   "metadata": {},
   "source": [
    "# 📊 Estado de Dados Brasil: Dashboards Executivos e SPECS Analíticas\n",
    "\n",
    "Este notebook consome a Fonte da Verdade (SOT) `base_consolidada.parquet` para gerar SPECS analíticas e responder de forma executiva a 5 perguntas estratégicas de negócio[cite: 3]:\n",
    "1. Qual é o cenário de diversidade de gênero nas carreiras de dados?[cite: 3]\n",
    "2. Quais tecnologias apresentam maior adoção entre os profissionais?[cite: 3]\n",
    "3. Qual é o índice de adoção de Inteligência Artificial e seu impacto?[cite: 3]\n",
    "4. Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?[cite: 3]\n",
    "5. Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e Inteligência Artificial?[cite: 3]"
   ]
  },
  {
   "cell_type": "code",
   "id": "etl",
   "metadata": {},
   "source": [
    "import duckdb\n",
    "import pandas as pd\n",
    "import re\n",
    "from pathlib import Path\n",
    "import plotly.graph_objects as go\n",
    "import plotly.express as px\n",
    "from plotly.subplots import make_subplots\n",
    "import sqlglot\n",
    "\n",
    "# 1. Carregamento da Fonte da Verdade (SOT)\n",
    "BASE_PROJETO = Path.cwd()\n",
    "ORIGEM_SOT = BASE_PROJETO / 'dados' / 'base_consolidada.parquet'\n",
    "SAIDA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'\n",
    "\n",
    "if not ORIGEM_SOT.exists():\n",
    "    BASE_PROJETO = Path('d:/d/pos/Fase 3')\n",
    "    ORIGEM_SOT = BASE_PROJETO / 'dados' / 'base_consolidada.parquet'\n",
    "    SAIDA_SPECS = BASE_PROJETO / 'dados' / 'bases_analiticas'\n",
    "\n",
    "assert ORIGEM_SOT.exists(), f'SOT não encontrada: {ORIGEM_SOT}'\n",
    "SAIDA_SPECS.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "df = pd.read_parquet(ORIGEM_SOT)\n",
    "df['id'] = df['id'].astype('string')\n",
    "df['respondente_key'] = df['ano_pesquisa'].astype('string') + '_' + df['id'].astype('string')\n",
    "\n",
    "# 2. Funções auxiliares\n",
    "def ativo(valor):\n",
    "    return not pd.isna(valor) and str(valor).strip().lower() in {'1', '1.0', 'true', 'sim', 'yes', 'x'}\n",
    "\n",
    "def itens_multivalorados(valor):\n",
    "    if pd.isna(valor):\n",
    "        return []\n",
    "    invalidos = {'', 'nan', 'não utilizo nenhuma das linguagens listadas', 'nenhuma', 'não se aplica'}\n",
    "    itens = [re.sub(r'\\s+', ' ', item).strip() for item in str(valor).split(',')]\n",
    "    return list(dict.fromkeys(item for item in itens if item.lower() not in invalidos))\n",
    "\n",
    "# 3. Construção das SPECS\n",
    "dimensoes = [\n",
    "    'respondente_key', 'ano_pesquisa', 'genero', 'regiao_onde_mora', \n",
    "    'nivel', 'faixa_salarial', 'modelo_trabalho_atual'\n",
    "]\n",
    "dimensoes = [c for c in dimensoes if c in df.columns]\n",
    "spec_respondentes = df[dimensoes].drop_duplicates('respondente_key').copy()\n",
    "\n",
    "familias_tecnologia = {\n",
    "    'linguagem': ['sql', 'python', 'r', 'java', 'scala', 'javascript'],\n",
    "    'banco_ou_plataforma': ['postgresql', 'mysql', 'sql_server', 'oracle', 'google_bigquery', 'snowflake', 'databricks'],\n",
    "    'cloud': ['aws_cloud', 'gcp_cloud', 'azure_cloud'],\n",
    "}\n",
    "\n",
    "linhas_tecnologia = []\n",
    "for familia, indicadores in familias_tecnologia.items():\n",
    "    for coluna in indicadores:\n",
    "        if coluna in df.columns:\n",
    "            base = df.loc[df[coluna].map(ativo), ['respondente_key', 'ano_pesquisa']].copy()\n",
    "            base['familia'] = familia\n",
    "            base['tecnologia'] = coluna.replace('_cloud', '').replace('_', ' ').strip().title()\n",
    "            linhas_tecnologia.append(base)\n",
    "\n",
    "spec_adocao_tecnologias = pd.concat(linhas_tecnologia, ignore_index=True).drop_duplicates()\n",
    "\n",
    "indicadores_ia = [\n",
    "    'ia_independente_descentralizada', 'ia_direcionamento_centralizado', \n",
    "    'utiliza_chatgpt_llms', 'usa_copilot', 'falta_expertise_recursos', \n",
    "    'dados_empresa_nao_prontos_ia', 'roi_nao_comprovado_ia'\n",
    "]\n",
    "barreiras_ia = {'falta_expertise_recursos', 'dados_empresa_nao_prontos_ia', 'roi_nao_comprovado_ia'}\n",
    "\n",
    "linhas_ia = []\n",
    "for coluna in indicadores_ia:\n",
    "    if coluna in df.columns:\n",
    "        base = df.loc[df[coluna].map(ativo), ['respondente_key', 'ano_pesquisa']].copy()\n",
    "        base['indicador_ia'] = coluna\n",
    "        base['tipo_barreira'] = coluna if coluna in barreiras_ia else pd.NA\n",
    "        linhas_ia.append(base)\n",
    "\n",
    "spec_adocao_ia = pd.concat(linhas_ia, ignore_index=True)\n",
    "spec_adocao_ia_uso = spec_adocao_ia[spec_adocao_ia['tipo_barreira'].isna()].copy()\n",
    "\n",
    "# 4. DuckDB e Agregações base\n",
    "con = duckdb.connect()\n",
    "con.register(\"spec_respondentes\", spec_respondentes)\n",
    "con.register(\"spec_adocao_tecnologias\", spec_adocao_tecnologias)\n",
    "con.register(\"spec_adocao_ia\", spec_adocao_ia)\n",
    "con.register(\"spec_adocao_ia_uso\", spec_adocao_ia_uso)\n",
    "\n",
    "def executar_sql(consulta):\n",
    "    return con.sql(sqlglot.parse_one(consulta, read=\"duckdb\").sql(dialect=\"duckdb\")).df()\n",
    "\n",
    "total_resp = int(executar_sql(\"SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes\").iloc[0, 0])\n",
    "print(f\"✅ Processamento concluído! Respondentes únicos: {total_resp:,}\")"
   ],
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "id": "resp1",
   "metadata": {},
   "source": [
    "## 🎯 Respostas: Cenário de Diversidade e Base Tecnológica\n",
    "\n",
    "* **P1. Qual é o cenário de diversidade de gênero nas carreiras de dados?**\n",
    "A análise revela um pipeline de talentos historicamente concentrado. As mulheres representam apenas **23,5%** da força de trabalho atual em dados[cite: 3]. \n",
    "\n",
    "* **P2. Quais tecnologias apresentam maior adoção entre os profissionais?**\n",
    "Existe uma base técnica clara: O **SQL** (57,6%) lidera de forma absoluta, seguido de perto pelo **Python** (54,9%)[cite: 3]. As plataformas Cloud como **AWS** e bancos como **PostgreSQL** formam o núcleo da arquitetura do mercado."
   ]
  },
  {
   "cell_type": "code",
   "id": "dash1",
   "metadata": {},
   "source": [
    "# Dados\n",
    "genero = executar_sql(\"\"\"\n",
    "    SELECT COALESCE(genero, 'Não informado') AS genero, COUNT(DISTINCT respondente_key) AS qtd\n",
    "    FROM spec_respondentes GROUP BY 1 ORDER BY qtd DESC\n",
    "\"\"\")\n",
    "top_tech = executar_sql(\"\"\"\n",
    "    SELECT tecnologia, COUNT(DISTINCT respondente_key) AS qtd, \n",
    "    ROUND(100.0 * COUNT(DISTINCT respondente_key) / (SELECT COUNT(DISTINCT respondente_key) FROM spec_respondentes), 1) AS pct\n",
    "    FROM spec_adocao_tecnologias GROUP BY tecnologia ORDER BY qtd DESC LIMIT 8\n",
    "\"\"\")\n",
    "\n",
    "# Visualização\n",
    "fig = make_subplots(rows=1, cols=2, specs=[[{\"type\": \"pie\"}, {\"type\": \"bar\"}]],\n",
    "                    subplot_titles=('Distribuição de Gênero', 'Top Tecnologias Mais Adotadas (%)'))\n",
    "\n",
    "fig.add_trace(go.Pie(labels=genero['genero'], values=genero['qtd'], hole=0.5, \n",
    "                     marker_colors=['#2a3f5f', '#d9485f', '#C8D4E3']), row=1, col=1)\n",
    "\n",
    "fig.add_trace(go.Bar(x=top_tech['pct'], y=top_tech['tecnologia'], orientation='h', \n",
    "                     marker_color='#0b7285', text=top_tech['pct'].apply(lambda x: f'{x}%'),\n",
    "                     textposition='outside'), row=1, col=2)\n",
    "\n",
    "fig.update_layout(template='plotly_white', height=450, showlegend=False, \n",
    "                  title={'text': '<b>Panorama Inicial: Quem são e o que usam?</b>', 'font': {'size': 18}})\n",
    "fig.update_yaxes(autorange=\"reversed\", row=1, col=2)\n",
    "fig.update_xaxes(range=[0, 70], row=1, col=2)\n",
    "fig.show()"
   ],
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "id": "resp2",
   "metadata": {},
   "source": [
    "## 🎯 Resposta: Índice de Adoção de Inteligência Artificial\n",
    "\n",
    "* **P3. Qual é o índice de adoção de IA e seu impacto?**\n",
    "O índice consolidado de adoção de IA na rotina dos profissionais é de **64,3%**[cite: 3]. O impacto operacional é notável pela intensidade: profissionais que utilizam IA não dependem de apenas uma ferramenta, aplicando, em média, **2,68** indicadores ou soluções[cite: 3] em seu fluxo de trabalho."
   ]
  },
  {
   "cell_type": "code",
   "id": "dash2",
   "metadata": {},
   "source": [
    "ia_adotantes = spec_adocao_ia_uso[\"respondente_key\"].nunique()\n",
    "intensidade_media = spec_adocao_ia_uso.groupby(\"respondente_key\")[\"indicador_ia\"].nunique().mean()\n",
    "pct_ia = (ia_adotantes / total_resp) * 100\n",
    "\n",
    "# Evolução IA\n",
    "evolucao = executar_sql(\"\"\"\n",
    "    SELECT ano_pesquisa::VARCHAR as ano,\n",
    "    ROUND(100.0 * COUNT(DISTINCT a.respondente_key) / COUNT(DISTINCT r.respondente_key), 1) as adocao_pct\n",
    "    FROM spec_respondentes r\n",
    "    LEFT JOIN spec_adocao_ia_uso a ON r.respondente_key = a.respondente_key\n",
    "    GROUP BY ano_pesquisa ORDER BY ano_pesquisa\n",
    "\"\"\")\n",
    "\n",
    "fig = make_subplots(rows=1, cols=2, specs=[[{\"type\": \"indicator\"}, {\"type\": \"scatter\"}]])\n",
    "\n",
    "# KPI\n",
    "fig.add_trace(go.Indicator(\n",
    "    mode=\"number+gauge\", value=pct_ia, number={'suffix': \"%\", 'font': {'size': 50, 'color': '#f08c46'}},\n",
    "    title={'text': f\"Profissionais utilizando IA<br><span style='font-size:14px;color:gray'>Média de {intensidade_media:.2f} ferramentas/usuário</span>\"},\n",
    "    gauge={'axis': {'range': [0, 100]}, 'bar': {'color': \"#f08c46\"}, 'bgcolor': \"lightgray\"}\n",
    "), row=1, col=1)\n",
    "\n",
    "# Linha de Evolução\n",
    "fig.add_trace(go.Scatter(x=evolucao['ano'], y=evolucao['adocao_pct'], mode='lines+markers+text',\n",
    "                         text=evolucao['adocao_pct'].apply(lambda x: f'{x}%'), textposition='top center',\n",
    "                         line=dict(color='#0b7285', width=4), marker=dict(size=10)), row=1, col=2)\n",
    "\n",
    "fig.update_layout(template='plotly_white', height=400, title={'text': '<b>Tração de IA no Fluxo de Trabalho</b>', 'font': {'size': 18}})\n",
    "fig.update_yaxes(range=[0, 100], title=\"% Adoção\", row=1, col=2)\n",
    "fig.show()"
   ],
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "id": "resp3",
   "metadata": {},
   "source": [
    "## 🎯 Resposta: Segmentação de Adoção\n",
    "\n",
    "* **P4. Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?**\n",
    "Sim. Os dados demonstram que a adoção de IA não é homogênea. \n",
    "Modelos de trabalho com maior flexibilidade (como **Remoto** e **Híbrido**) e profissionais com cargos mais altos (**Especialistas/Seniores**) apresentam taxas de adoção significativamente mais elevadas. Isso evidencia que a maturidade profissional e a autonomia do regime de trabalho aceleram a experimentação tecnológica."
   ]
  },
  {
   "cell_type": "code",
   "id": "dash3",
   "metadata": {},
   "source": [
    "def query_segmento(coluna):\n",
    "    return executar_sql(f\"\"\"\n",
    "        SELECT COALESCE({coluna}, 'N/I') as segmento, \n",
    "               COUNT(DISTINCT r.respondente_key) as total,\n",
    "               COUNT(DISTINCT a.respondente_key) as adotantes,\n",
    "               ROUND(100.0 * COUNT(DISTINCT a.respondente_key) / COUNT(DISTINCT r.respondente_key), 1) as pct\n",
    "        FROM spec_respondentes r\n",
    "        LEFT JOIN spec_adocao_ia_uso a ON r.respondente_key = a.respondente_key\n",
    "        GROUP BY 1 HAVING COUNT(DISTINCT r.respondente_key) > 50 ORDER BY pct ASC\n",
    "    \"\"\")\n",
    "\n",
    "df_reg = query_segmento('regiao_onde_mora')\n",
    "df_niv = query_segmento('nivel')\n",
    "df_mod = query_segmento('modelo_trabalho_atual')\n",
    "\n",
    "# Padronizando nomes longos\n",
    "df_mod['segmento'] = df_mod['segmento'].replace({\n",
    "    'Modelo 100% presencial': 'Presencial',\n",
    "    'Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)': 'Híbrido Flexível',\n",
    "    'Modelo híbrido com dias fixos de trabalho presencial': 'Híbrido Fixo',\n",
    "    'Modelo 100% remoto': 'Remoto'\n",
    "})\n",
    "\n",
    "fig = make_subplots(rows=1, cols=3, subplot_titles=('Por Região', 'Por Senioridade', 'Por Modelo de Trabalho'),\n",
    "                    horizontal_spacing=0.1)\n",
    "\n",
    "def add_bar(df, row, col):\n",
    "    fig.add_trace(go.Bar(\n",
    "        x=df['pct'], y=df['segmento'], orientation='h', marker_color='#2a3f5f',\n",
    "        text=df.apply(lambda row: f\"{row['pct']}% (n={row['total']})\", axis=1), textposition='inside', insidetextanchor='end'\n",
    "    ), row=row, col=col)\n",
    "\n",
    "add_bar(df_reg, 1, 1)\n",
    "add_bar(df_niv, 1, 2)\n",
    "add_bar(df_mod, 1, 3)\n",
    "\n",
    "fig.update_layout(template='plotly_white', height=450, showlegend=False, title={'text': '<b>Adoção de IA por Demografia e Rotina</b>', 'font': {'size': 18}})\n",
    "fig.update_xaxes(range=[0, 100])\n",
    "fig.show()"
   ],
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "id": "resp4",
   "metadata": {},
   "source": [
    "## 🎯 Resposta: Oportunidades e Desafios para Investimento\n",
    "\n",
    "* **P5. Quais oportunidades e desafios podem ser identificados para empresas que desejam investir em Dados e IA?**[cite: 3]\n",
    "* **O Desafio Central:** A transição da fase \"experimental\" para a \"escala produtiva\". O principal gargalo não é o interesse da equipe, mas a **Falta de Expertise Técnica**, a **Desorganização dos Dados da Empresa** e a dificuldade de provar o **ROI**[cite: 3].\n",
    "* **A Oportunidade:** Existe uma clareza absoluta de qual é o foco inicial: **Capacitação**. Investir pesadamente em treinar o time nas bases sólidas identificadas (SQL, Python e AWS/GCP), acompanhado de programas de **Governança de Dados**. Times organizados e bem treinados nas ferramentas líderes são o único caminho para fazer a IA gerar valor financeiro escalável[cite: 2, 3]."
   ]
  },
  {
   "cell_type": "code",
   "id": "dash4",
   "metadata": {},
   "source": [
    "resumo_diretoria = pd.DataFrame([\n",
    "    {\"Vetor Estratégico\": \"Pessoas & Cultura\", \"Fato Observado\": \"Mulheres representam 23.5% do mercado.\", \"Ação Recomendada\": \"Auditar processos de contratação e retenção. Fomentar lideranças femininas em dados.\"},\n",
    "    {\"Vetor Estratégico\": \"Stack Tecnológico\", \"Fato Observado\": \"SQL (58%) e Python (55%) ditam o ritmo de engenharia e análise.\", \"Ação Recomendada\": \"Padronizar trilhas de treinamento focadas nestas linguagens para nivelar o time.\"},\n",
    "    {\"Vetor Estratégico\": \"Adoção de IA\", \"Fato Observado\": \"64.3% já experimentam IA, mas esbarram na 'Falta de Expertise' e 'Dados Desorganizados'.\", \"Ação Recomendada\": \"Pausar adoção de ferramentas da moda; focar orçamento em Governança de Dados e letramento corporativo em IA.\"}\n",
    "])\n",
    "\n",
    "fig = go.Figure(data=[go.Table(\n",
    "    columnorder=[1, 2, 3],\n",
    "    columnwidth=[20, 35, 45],\n",
    "    header=dict(values=list(resumo_diretoria.columns), fill_color='#2a3f5f', font=dict(color='white', size=14), align='left'),\n",
    "    cells=dict(values=[resumo_diretoria[k] for k in resumo_diretoria.columns], fill_color='#F4F6F9', align='left', font=dict(size=13), height=40)\n",
    ")])\n",
    "\n",
    "fig.update_layout(title={'text': '<b>Plano de Ação para Diretoria (Takeaways)</b>', 'font': {'size': 18}}, height=300, margin=dict(t=50, b=0))\n",
    "fig.show()"
   ],
   "outputs": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}